In [2]:
import os
import json
import requests
from pymilvus import MilvusClient, DataType

# milvus 连接配置
MILVUS_URL = "http://localhost:19530"
DB_NAME = "rag_tutorial"
COLLECTION_NAME = "docs"
KNOWLEDGE_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "knowledge.txt")

# 嵌入模型
EMBED_MODEL_NAME = "Pro/BAAI/bge-m3"
EMBED_DIM = 1024  # BGE-M3 输出的向量维度

# --- API 密钥 ---
# 优先从环境变量读取，也可直接在代码中填写
# SiliconFlow 注册: https://cloud.siliconflow.cn（新用户有免费额度）
SILICONFLOW_API_KEY = os.getenv("SILICONFLOW_API_KEY", "")
SILICONFLOW_BASE_URL = "https://api.siliconflow.cn/v1"

# --- LLM（大语言模型）配置 ---
# 使用阿里云 DashScope（通义千问）的 Qwen 模型
# 通过 OpenAI 兼容接口调用，base_url 和 api_key 从环境变量读取
LLM_MODEL = "qwen3.7-max"
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY", "")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL", "")

# 文本切分参数
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50


# 一、文件的加载和切分
def load_and_spilit(file_path, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    chunks = []
    start = 0
    while text < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
            start += chunk_size - chunk_overlap
    print(f"文档加载完成，共切分了{len(chunks)}个文本块")
    return chunks


# 二、文本向量化
def embed_texts(texts):
    """
    批量将文本列表转换为向量列表

    参数:
        texts: 文本块列表 ["文本1", "文本2", ...]

    返回:
        all_vectors: 向量列表 [[0.12, -0.34, ...], [0.56, 0.78, ...], ...]

    调用 SiliconFlow 的 /embeddings API，每次最多处理 20 条
    """
    url = f"{SILICONFLOW_BASE_URL}/embeddings"
    headers = {
        "Authorization": f"Bearer {SILICONFLOW_API_KEY}",  # API 鉴权
        "Content-Type": "application/json"
    }

    all_vectors = []
    batch_size = 20  # API 限制，每次最多发 20 条文本

    # 分批处理，每批 batch_size 条
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]  # 取出当前批次

        # 构造 API 请求体
        payload = {
            "model": EMBED_MODEL_NAME,  # 指定使用 BGE-M3 模型
            "input": batch,  # 要向量化的文本列表
            "encoding_format": "float"  # 返回浮点数格式的向量
        }

        # 发送 HTTP POST 请求到 SiliconFlow
        resp = requests.post(url, headers=headers, json=payload)
        resp.raise_for_status()  # 如果 HTTP 状态码不是 200，抛出异常
        data = resp.json()

        # 从响应中提取每条文本对应的向量
        # 响应格式: {"data": [{"embedding": [0.12, ...], "index": 0}, ...]}
        for item in data["data"]:
            all_vectors.append(item["embedding"])

        print(f"  ✅ 已向量化 {min(i + batch_size, len(texts))}/{len(texts)} 条")

    return all_vectors


def embed_query(text):
    """
    将单条查询文本转换为向量（用户提问时用）
    参数:
        text: 用户的问题文本，如 "atguigu助手有哪些套餐？"
    返回:
        一个 1024 维的向量列表 [0.12, -0.34, ...]
    和 embed_texts 调用的是同一个 API，只是 input 只有一条文本
    """
    url = f"{SILICONFLOW_BASE_URL}/embeddings"
    headers = {
        "Authorization": f"Bearer {SILICONFLOW_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": EMBED_MODEL_NAME,
        "input": [text],  # 注意：即使只有一条也要包在列表里
        "encoding_format": "float"
    }
    resp = requests.post(url, headers=headers, json=payload)
    resp.raise_for_status()
    # 取第一条（也是唯一一条）结果的向量
    return resp.json()["data"][0]["embedding"]





NameError: name '__file__' is not defined